![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 23 -- Lab 2: CLIP Zero-Shot and LoRA Fine-Tuning

**Scenario:** You are entering an image classification competition. The dataset has 102 classes of flowers -- far too many to label from scratch. But you have CLIP, a model that already understands both images and text. Can you classify flowers without ANY training? And how much better can you get with just a tiny amount of fine-tuning?

You will:
1. Use CLIP for **zero-shot classification** -- no training at all!
2. Improve accuracy with **prompt engineering** -- the art of asking CLIP the right question
3. Fine-tune CLIP with **LoRA** -- push accuracy from ~67% to ~90% with minimal compute
4. Build an **image retrieval system** -- search images with natural language

| What You'll Learn | Why It Matters |
|---|---|
| Zero-shot classification | Classify anything without training -- useful when you have no labeled data |
| Prompt engineering | Small text changes = big accuracy gains (free performance!) |
| Prompt ensembling | Combine multiple prompts for more robust predictions |
| LoRA fine-tuning of CLIP | Adapt a foundation model with 0.5% trainable parameters |
| Text-to-image retrieval | Build a search engine in 10 lines of code |

In [ ]:
!pip install open_clip_torch peft -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

import open_clip

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import json
import urllib.request

plt.rcParams['figure.dpi'] = 120

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

---
# Part 1 -- Load CLIP and the Dataset (GIVEN)

We use **OpenCLIP** (open-source CLIP) with a ViT-B/32 backbone. This model was trained on 400M image-text pairs from the internet -- it has seen descriptions of almost everything.

Our dataset is **Flowers-102**: 102 fine-grained flower species. This is a HARD dataset because many flowers look similar (different types of orchids, different types of daisies, etc.).

In [ ]:
# --- GIVEN: Load CLIP model ---

model_name = 'ViT-B-32'
pretrained = 'laion2b_s34b_b79k'

clip_model, _, preprocess = open_clip.create_model_and_transforms(
    model_name, pretrained=pretrained
)
tokenizer = open_clip.get_tokenizer(model_name)

clip_model = clip_model.to(device)
clip_model.eval()

print(f'Model: {model_name}')
print(f'Parameters: {sum(p.numel() for p in clip_model.parameters()):,}')
print(f'Image input size: 224x224')
print(f'Embedding dimension: 512')

In [ ]:
# --- GIVEN: Load Flowers-102 dataset ---

# Download class names
url = 'https://gist.githubusercontent.com/JosephKJ/94c7728ed1a8e0cd87fe6a029769cde1/raw/403325f5110cb0f3099734c5edb9f457539c77e9/Oxford-102_Flower_dataset_labels.txt'
urllib.request.urlretrieve(url, 'flower_labels.txt')

with open('flower_labels.txt') as f:
    class_names = [line.strip() for line in f.readlines()]

print(f'Number of classes: {len(class_names)}')
print(f'First 10 classes: {class_names[:10]}')
print(f'Last 10 classes:  {class_names[-10:]}')
print()

# Load datasets
flowers_train = torchvision.datasets.Flowers102(
    root='./data', split='train', download=True, transform=preprocess
)
flowers_val = torchvision.datasets.Flowers102(
    root='./data', split='val', download=True, transform=preprocess
)
flowers_test = torchvision.datasets.Flowers102(
    root='./data', split='test', download=True, transform=preprocess
)

print(f'Train: {len(flowers_train)} images')
print(f'Val:   {len(flowers_val)} images')
print(f'Test:  {len(flowers_test)} images')

In [ ]:
# --- GIVEN: Quick CLIP demo ---
# Show that CLIP can match images to text descriptions

# Load a raw image for display
flowers_raw = torchvision.datasets.Flowers102(
    root='./data', split='test', download=True,
    transform=transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
)

# Pick 4 test images
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    img_raw, label = flowers_raw[i * 500]
    ax.imshow(img_raw.permute(1, 2, 0))
    ax.set_title(f'True: {class_names[label]}', fontsize=9)
    ax.axis('off')
plt.suptitle('Sample flowers from the test set', fontsize=12)
plt.tight_layout()
plt.show()

---
# Part 2 -- Zero-Shot Classification

The idea is simple:
1. Encode all 102 class names as text embeddings
2. Encode a test image as an image embedding
3. Find which class name is most similar to the image

No training needed! CLIP already knows what these flowers look like because it saw image-text pairs on the internet.

In [ ]:
# ============================================================
# TASK 1: Zero-shot classification
# ============================================================
# Step 1: Create text embeddings for all 102 flower classes.
# Use the simple prompt: "a photo of a {class_name}"

# Create text prompts for each class
text_prompts = [f"a photo of a {name}" for name in class_names]

# Tokenize and encode
with torch.no_grad():
    text_tokens = tokenizer(text_prompts).to(device)
    text_embeddings = clip_model.encode_text(None)  # TODO: pass the tokens
    # Normalize to unit length (so dot product = cosine similarity)
    text_embeddings = F.normalize(text_embeddings, dim=None)  # TODO: which dim? (1)

print(f'Text embeddings shape: {text_embeddings.shape}')  # Should be (102, 512)

In [ ]:
# ============================================================
# TASK 2: Evaluate zero-shot accuracy on the test set
# ============================================================

test_loader = DataLoader(flowers_test, batch_size=128, shuffle=False, num_workers=2)

correct = 0
total = 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Zero-shot eval'):
        images = images.to(device)
        labels = labels.to(device)

        # Encode images
        image_embeddings = clip_model.encode_image(None)  # TODO: pass the images
        image_embeddings = F.normalize(image_embeddings, dim=None)  # TODO: which dim?

        # Compute similarity: each image vs all 102 class texts
        # (batch, 512) @ (512, 102) = (batch, 102)
        similarity = image_embeddings @ None  # TODO: text_embeddings transposed (.T)

        # Predict = class with highest similarity
        predictions = similarity.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

zero_shot_acc = correct / total
print(f'\nZero-shot accuracy: {zero_shot_acc*100:.1f}%')
print(f'(Random chance would be {100/102:.1f}%)')
print(f'Not bad for ZERO training on flowers!')

---
# Part 3 -- Prompt Engineering

The prompt makes a big difference! CLIP was trained on internet captions, so it responds better to natural descriptions than bare class names.

Compare:
- Bad: `"rose"` (could mean anything -- a person named Rose, the color, etc.)
- Good: `"a photo of a rose"` (clearly an image of the flower)
- Better: `"a close-up photo of a rose, a type of flower"` (more context)

**Prompt ensembling** takes this further: use multiple different prompts for each class, encode them all, and average the embeddings. This makes predictions more robust.

In [ ]:
# ============================================================
# TASK 3: Try different prompt templates
# ============================================================
# Test 3 different prompt styles and see which gives best accuracy.

prompt_templates = [
    "a photo of a {}",                              # basic
    "a photo of a {}, a type of flower",            # with context
    None,  # TODO: Write your own template! Try to be descriptive.
           # Ideas: "a close-up photo of a {} flower"
           #        "a botanical photograph of a {}"
           #        "a bright colorful {} in a garden"
]


def zero_shot_accuracy(prompt_template):
    """Compute zero-shot accuracy with a given prompt template."""
    prompts = [prompt_template.format(name) for name in class_names]
    with torch.no_grad():
        tokens = tokenizer(prompts).to(device)
        text_emb = F.normalize(clip_model.encode_text(tokens), dim=1)

    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            img_emb = F.normalize(clip_model.encode_image(images), dim=1)
            preds = (img_emb @ text_emb.T).argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


# Evaluate each template
print('Prompt template comparison:')
print('-' * 60)
template_results = {}
for template in prompt_templates:
    if template is None:
        continue
    acc = zero_shot_accuracy(template)
    template_results[template] = acc
    print(f'  "{template}"')
    print(f'  -> Accuracy: {acc*100:.1f}%')
    print()

In [ ]:
# ============================================================
# TASK 4: Prompt ensembling
# ============================================================
# Use MULTIPLE templates and AVERAGE the text embeddings.
# This gives more robust predictions than any single template.

ensemble_templates = [
    "a photo of a {}",
    "a photo of a {}, a type of flower",
    "a close-up photo of a {} flower",
    "a bright {} in a garden",
    "a botanical illustration of a {}",
]

# Encode all templates for each class and average
with torch.no_grad():
    all_text_embeddings = []
    for template in ensemble_templates:
        prompts = [template.format(name) for name in class_names]
        tokens = tokenizer(prompts).to(device)
        emb = F.normalize(clip_model.encode_text(tokens), dim=1)
        all_text_embeddings.append(emb)

    # Average across templates, then re-normalize
    # Stack: (5, 102, 512) -> mean over dim 0 -> (102, 512)
    ensemble_embeddings = torch.stack(all_text_embeddings).mean(dim=None)  # TODO: which dim to average? (0)
    ensemble_embeddings = F.normalize(ensemble_embeddings, dim=None)  # TODO: re-normalize (dim=1)

# Evaluate with ensemble
correct = 0
total = 0
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Ensemble eval'):
        images, labels = images.to(device), labels.to(device)
        img_emb = F.normalize(clip_model.encode_image(images), dim=1)
        preds = (img_emb @ ensemble_embeddings.T).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

ensemble_acc = correct / total
print(f'\nPrompt ensemble accuracy: {ensemble_acc*100:.1f}%')
print(f'Improvement over basic prompt: +{(ensemble_acc - zero_shot_acc)*100:.1f}%')
print(f'That\'s free performance -- no training, just better prompts!')

---
# Part 4 -- LoRA Fine-Tuning

Zero-shot is impressive, but 67% is not enough for a competition. Let's fine-tune CLIP on the Flowers-102 training set using **LoRA** (same technique from Day 19 and Day 20!).

LoRA adds tiny trainable matrices (rank-8) to the attention layers. Instead of updating all 151M parameters, we only train ~0.5% of them.

**Training approach:** We fine-tune the image encoder only. For each image, we compute its similarity to all 102 class text embeddings, and train it like a classification problem (cross-entropy on the similarities).

In [ ]:
# --- GIVEN: Load a fresh CLIP model for fine-tuning ---
# We use the transformers library version because PEFT integrates with it

from transformers import CLIPModel, CLIPProcessor
from peft import LoraConfig, get_peft_model

# Load fresh model
clip_ft = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print(f'Total parameters: {sum(p.numel() for p in clip_ft.parameters()):,}')

In [ ]:
# ============================================================
# TASK 5: Apply LoRA
# ============================================================
# Same pattern as Day 19 (ViT) and Day 20 (HuBERT)!
# Fill in the LoRA configuration.

lora_config = LoraConfig(
    r=None,              # TODO: rank of the low-rank matrices (try 8)
    lora_alpha=None,     # TODO: scaling factor (try 16)
    lora_dropout=None,   # TODO: dropout probability (try 0.1)
    target_modules=None, # TODO: which layers? (try ["q_proj", "v_proj"])
)

clip_ft = get_peft_model(clip_ft, lora_config)
clip_ft.print_trainable_parameters()

In [ ]:
# --- GIVEN: Prepare text embeddings for training ---
# We pre-compute the text embeddings once (they don't change during training)

text_inputs = processor(
    text=[f"a photo of a {name}, a type of flower" for name in class_names],
    return_tensors="pt", padding=True, truncation=True
).to(device)

with torch.no_grad():
    text_features = clip_ft.get_text_features(**text_inputs)
    text_features = F.normalize(text_features, dim=1)  # (102, 512)

print(f'Pre-computed text embeddings: {text_features.shape}')

In [ ]:
# --- GIVEN: Data loaders for fine-tuning ---

# Use the processor's image transforms for the HuggingFace CLIP model
from torchvision.transforms import Compose, Resize, CenterCrop, ToTensor, Normalize

clip_transform = Compose([
    Resize(224),
    CenterCrop(224),
    ToTensor(),
    Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
              std=[0.26862954, 0.26130258, 0.27577711]),
])

train_dataset = torchvision.datasets.Flowers102(
    root='./data', split='train', download=True, transform=clip_transform
)
val_dataset = torchvision.datasets.Flowers102(
    root='./data', split='val', download=True, transform=clip_transform
)
test_dataset = torchvision.datasets.Flowers102(
    root='./data', split='test', download=True, transform=clip_transform
)

# Combine train + val for more training data
from torch.utils.data import ConcatDataset
train_full = ConcatDataset([train_dataset, val_dataset])

train_loader_ft = DataLoader(train_full, batch_size=64, shuffle=True, num_workers=2)
test_loader_ft = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f'Training images: {len(train_full)}')
print(f'Test images: {len(test_dataset)}')

In [ ]:
# ============================================================
# TASK 6: Training loop
# ============================================================
# Fill in the TODOs. This is the same pattern as Day 20!

optimizer = optim.AdamW(clip_ft.parameters(), lr=1e-4, weight_decay=0.01)
num_epochs = 5
losses = []

clip_ft.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader_ft, desc=f'Epoch {epoch+1}/{num_epochs}', leave=False):
        images = images.to(device)
        labels = labels.to(device)

        # Get image features from the model
        image_features = clip_ft.get_image_features(pixel_values=None)  # TODO: pass images
        image_features = F.normalize(image_features, dim=1)

        # Compute similarity to all 102 class text embeddings
        # logits shape: (batch, 102)
        logit_scale = clip_ft.logit_scale.exp()
        logits = logit_scale * (image_features @ text_features.T)

        # Classification loss
        loss = F.cross_entropy(None, None)  # TODO: (logits, labels)

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = epoch_loss / len(train_loader_ft)
    train_acc = correct / total
    losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{num_epochs} | Loss: {avg_loss:.3f} | Train Acc: {train_acc*100:.1f}%')

print('\nFine-tuning complete!')

In [ ]:
# --- GIVEN: Plot training loss ---
plt.figure(figsize=(7, 4))
plt.plot(range(1, num_epochs + 1), losses, 'b-o', linewidth=2, markersize=8)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('CLIP + LoRA Fine-Tuning on Flowers-102')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
# Part 5 -- Evaluation and Comparison

Let's see how much our LoRA fine-tuning improved things!

In [ ]:
# ============================================================
# TASK 7: Evaluate the fine-tuned model on the test set
# ============================================================

clip_ft.eval()
correct = 0
total = 0

with torch.no_grad():
    # Re-compute text features with the fine-tuned model
    text_features_ft = clip_ft.get_text_features(**text_inputs)
    text_features_ft = F.normalize(text_features_ft, dim=1)

    for images, labels in tqdm(test_loader_ft, desc='Test eval'):
        images = images.to(device)
        labels = labels.to(device)

        # Get image features
        image_features = clip_ft.get_image_features(pixel_values=None)  # TODO: pass images
        image_features = F.normalize(image_features, dim=1)

        # Compute similarity and predict
        logits = image_features @ None  # TODO: text_features_ft.T
        preds = logits.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

lora_acc = correct / total
print(f'\nLoRA fine-tuned accuracy: {lora_acc*100:.1f}%')

In [ ]:
# --- GIVEN: Comparison bar chart ---

results = {
    'Zero-shot\n(basic prompt)': zero_shot_acc,
    'Prompt\nEnsemble': ensemble_acc,
    'LoRA\nFine-tuned': lora_acc,
}

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#636e72', '#0984e3', '#00b894']
bars = ax.bar(results.keys(), [v*100 for v in results.values()],
              color=colors, edgecolor='black', width=0.6)

ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Flowers-102 Classification: Zero-shot vs Prompt Engineering vs LoRA', fontsize=12)
ax.set_ylim(0, 100)
ax.axhline(y=100/102, color='gray', linestyle=':', alpha=0.5, label='Random chance')

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nSummary:')
print(f'  Zero-shot:       {zero_shot_acc*100:.1f}% (no training at all)')
print(f'  Prompt ensemble: {ensemble_acc*100:.1f}% (no training, just better prompts)')
print(f'  LoRA fine-tuned: {lora_acc*100:.1f}% (5 epochs, 0.5% params trained)')
print(f'\n  Improvement from LoRA: +{(lora_acc - zero_shot_acc)*100:.1f}% over zero-shot!')

---
# Part 6 -- Bonus: Image Retrieval

CLIP embeddings are also perfect for **image search**: describe what you want in words, and find the most similar images. This is how Google Images, Pinterest, and Apple Photos work under the hood.

In [ ]:
# ============================================================
# TASK 8: Text-to-image retrieval
# ============================================================
# Search the test set using a natural language query.

# Step 1: Pre-compute all test image embeddings
all_image_features = []
with torch.no_grad():
    for images, _ in tqdm(test_loader_ft, desc='Indexing images'):
        images = images.to(device)
        feats = clip_ft.get_image_features(pixel_values=images)
        feats = F.normalize(feats, dim=1)
        all_image_features.append(feats.cpu())

all_image_features = torch.cat(all_image_features, dim=0)  # (6149, 512)
print(f'Indexed {all_image_features.shape[0]} images')

# Step 2: Search with a text query
query = "a bright red flower with many petals"  # Try changing this!

with torch.no_grad():
    query_inputs = processor(text=[query], return_tensors="pt", padding=True).to(device)
    query_features = clip_ft.get_text_features(**query_inputs)
    query_features = F.normalize(query_features, dim=1).cpu()

# Step 3: Find top-5 most similar images
similarities = (query_features @ all_image_features.T).squeeze(0)
top5_indices = similarities.topk(5).indices

# Step 4: Display results
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
fig.suptitle(f'Query: "{query}"', fontsize=13)

flowers_display = torchvision.datasets.Flowers102(
    root='./data', split='test',
    transform=transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
)

for i, (ax, idx) in enumerate(zip(axes, top5_indices)):
    img, label = flowers_display[idx.item()]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(f'#{i+1}: {class_names[label]}\nsim={similarities[idx]:.3f}', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

---
# Discussion Questions

1. **Competition strategy:** You have a new dataset with 50 classes and 20 labeled examples per class. Would you: (a) train from scratch, (b) use CLIP zero-shot, or (c) use CLIP + LoRA? Why?

2. **Prompt engineering is free performance.** In a competition, you have 5 minutes left. You can either: tune a hyperparameter or write better prompts. Which do you pick and why?

3. **LoRA rank trade-off:** What happens if you use r=1 (tiny) vs r=64 (large)? Think about: expressiveness, overfitting risk, training speed, memory.

4. **Image vs text encoder:** We only fine-tuned attention layers in both encoders. What if we only fine-tuned the image encoder? What about only the text encoder?

5. **When does CLIP fail?** Think of a domain where CLIP zero-shot would perform poorly. (Hint: what kinds of images were probably NOT in its 400M training pairs?)

---
# Wrap-Up

| Concept | What You Learned |
|---|---|
| Zero-shot classification | Encode class names as text, compare to image embeddings -- no training! |
| Prompt engineering | Better text descriptions = better accuracy (free!) |
| Prompt ensembling | Average multiple prompt embeddings for robustness |
| LoRA fine-tuning | Train 0.5% of parameters, get massive accuracy boost |
| Image retrieval | Same embeddings power text-to-image search |
| Competition strategy | Start zero-shot, improve with prompts, fine-tune if data available |

**Key takeaway for competitions:** Always try CLIP zero-shot first as a baseline. It's free, fast, and often surprisingly good. Then improve with prompt engineering (also free). Only fine-tune if you have training data and it's worth the compute.